# Evolving `AIEngineerNewsLetter` with `rsi-cli`

An interactive, reproducible walkthrough of **spec- and metric-driven Recursive Self-Improvement (RSI)** using [`rsi-cli`](../../README.md) on a real TypeScript monorepo: [`DontAskFuture/AIEngineerNewsLetter`](https://github.com/DontAskFuture/AIEngineerNewsLetter).

---

## 1. Problem Overview

`AIEngineerNewsLetter` aggregates model launches from OpenRouter alongside crawled official blog and press announcements. Its shared package exports:

```text
packages/shared/src/model-releases.ts : mergeWatchedModelReleases(openRouterReleases, documentReleases, options)
```

This function enriches OpenRouter entries with official announcement URLs, deduplicates remaining announcements by significant model-token overlap, and returns the top `limit` chronologically ordered releases. In the baseline commit (`3724cb21`), nested `O(D × R)` matching, repeated Unicode regex tokenization, `O(D²)` announcement deduplication, and full-array cloning/sorting leave throughput at **`141.18 ops/sec`** (`p95 = 7.59 ms`).

### How `rsi-cli` Evolves It Safely Out-of-Tree

| Component | Configuration in `examples/ai-newsletter-eval/rsi.yaml` |
|---|---|
| **Target Repository** | `https://github.com/DontAskFuture/AIEngineerNewsLetter.git` (`main`, TypeScript) |
| **Target Function** | `packages/shared/src/model-releases.ts:mergeWatchedModelReleases` |
| **Eval Wrapper** | Harness-managed `ts_fn` Golden-Master (`equivalence: exact_output`, `warmup_iters: 4`, `measure_iters: 30`) |
| **Correctness Blockers** | `golden_equivalence`, `shared_unit_tests` (`pnpm --dir packages/shared test`), `shared_typecheck` (`pnpm --dir packages/shared typecheck`) |
| **Protected Paths** | `**/*.test.ts`, `packages/db/drizzle/**`, `packages/shared/package.json`, `packages/shared/tsconfig.json` (**61 files** locked read-only `0o444` + SHA-256 verified) |
| **Optimization Goals** | `ops_per_sec` (`MAXIMIZE`, `target_ratio: 2.0`, weight `0.7`) & `p95_ms` (`MINIMIZE`, weight `0.3`) |
| **Recorded Outcome** | **`GOAL_REACHED` on Step 1 of 3** — **`45.64×`** throughput (`141.18 → 6,443.00 ops/sec`), **`39.42×`** lower p95 latency (`7.59 ms → 0.19 ms`), `0` repair turns, `0` reverts |

### Walkthrough Roadmap
1. **Problem Overview** *(this section)*
2. **Portable Setup & `rsi doctor` Preflight** — discover `rsi` and verify local runtimes, drivers, and GitHub auth live
3. **Interactive Configuration & Dataset Explorer** — inspect `rsi.yaml` & `inputs.jsonl` and tune run parameters
4. **Zero-LLM Preflight Dry Run (`rsi evolve --dry-run`)** — validate Step-0 Golden-Master capture, blockers, and IQR noise calibration
5. **Evolution Runner & Recorded Reference Replay** — replay the real `run_20261123_042237_1dd900` result or flip one flag to run live
6. **Interactive Visual Scorecard & Charts** — inline SVG throughput, latency, per-trial spread, and blocker duration charts
7. **Clean Net Diff & Algorithmic Breakdown (`rsi diff`)** — inspect why the baseline was `O(D × R)` + `O(D²)` and how Step 1 achieved `45.64×`
8. **Toggleable GitHub Delivery (`rsi publish`)** — preview or execute PR-based (`pull-request --auto-merge`), Zero-PR (`zero-pr`), or `bundle-only` delivery


## 2. Portable Setup & `rsi doctor` Preflight

This cell resolves the repository root portably (whether you launched Jupyter from the repository root or `examples/ai-newsletter-eval/`), locates the `rsi` binary (`target/release/rsi` → `target/debug/rsi` → `PATH`), and runs `rsi doctor --timeout-sec 2` live with sanitized output paths.


In [1]:
from __future__ import annotations

import html
import json
import os
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

from IPython.display import HTML, display


def find_repo_root() -> Path:
    """Locate the rsi-cli repository root from cwd or RSI_CLI_ROOT."""
    env_root = os.environ.get("RSI_CLI_ROOT")
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if (candidate / "examples" / "ai-newsletter-eval" / "rsi.yaml").is_file():
            return candidate

    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "examples" / "ai-newsletter-eval" / "rsi.yaml").is_file():
            return candidate
        if (candidate / "rsi.yaml").is_file() and (candidate / "inputs.jsonl").is_file():
            parent_root = candidate.parent.parent
            if (parent_root / "Cargo.toml").is_file():
                return parent_root
            return candidate
    raise FileNotFoundError(
        "Could not locate examples/ai-newsletter-eval/rsi.yaml. "
        "Launch Jupyter from the rsi-cli repo root or examples/ai-newsletter-eval/, "
        "or set RSI_CLI_ROOT."
    )


REPO_ROOT = find_repo_root()
EVAL_DIR = (
    REPO_ROOT / "examples" / "ai-newsletter-eval"
    if (REPO_ROOT / "examples" / "ai-newsletter-eval" / "rsi.yaml").is_file()
    else REPO_ROOT
)
CONFIG_PATH = EVAL_DIR / "rsi.yaml"
INPUTS_PATH = EVAL_DIR / "inputs.jsonl"


def sanitize_text(text: str) -> str:
    """Replace machine-local repo and home paths with portable placeholders."""
    out = text.replace(str(REPO_ROOT), "$REPO_ROOT")
    home_dir = str(Path.home())
    if home_dir:
        out = out.replace(home_dir, "~")
    return out


def rel_display(path: Path) -> str:
    """Render a path relative to REPO_ROOT when possible."""
    try:
        return str(path.resolve().relative_to(REPO_ROOT))
    except ValueError:
        return sanitize_text(str(path))


def locate_rsi_binary() -> tuple[str | None, str]:
    """Prefer target/release/rsi, then target/debug/rsi, then PATH."""
    for rel_candidate in (Path("target/release/rsi"), Path("target/debug/rsi")):
        full = REPO_ROOT / rel_candidate
        if full.is_file() and os.access(full, os.X_OK):
            return str(full), str(rel_candidate)
    on_path = shutil.which("rsi")
    if on_path:
        return on_path, "rsi (PATH)"
    return None, "not found"


RSI_BIN, RSI_BIN_LABEL = locate_rsi_binary()
print(f"Repository layout : $REPO_ROOT (config: {rel_display(CONFIG_PATH)})")
print(f"Resolved rsi CLI  : {RSI_BIN_LABEL}")

if RSI_BIN is None:
    print(
        "\nBuild hint: run `cargo build --release` (or `cargo install --path .`) "
        "from $REPO_ROOT, then re-run this cell."
    )
else:
    proc = subprocess.run(
        [RSI_BIN, "doctor", "--timeout-sec", "2"],
        stdin=subprocess.DEVNULL,
        capture_output=True,
        text=True,
        check=False,
    )
    doctor_stdout = sanitize_text(proc.stdout.strip())
    doctor_stderr = sanitize_text(proc.stderr.strip())
    print("\n--- Live `rsi doctor --timeout-sec 2` ---")
    print(doctor_stdout)
    if doctor_stderr:
        print(f"\n[stderr] {doctor_stderr}")

    ok_count = doctor_stdout.count("[OK]")
    skip_count = doctor_stdout.count("[SKIP]")
    fail_count = doctor_stdout.count("[FAIL]")
    status_color = "#15803d" if proc.returncode == 0 else "#b91c1c"
    status_label = "READY" if proc.returncode == 0 else f"ISSUES DETECTED (exit {proc.returncode})"

    display(
        HTML(
            f"""
            <div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
                        background: #ffffff; color: #1f2937; border: 1px solid #e5e7eb;
                        border-left: 4px solid {status_color}; border-radius: 8px;
                        padding: 12px 16px; margin: 8px 0; max-width: 780px;">
              <div style="display: flex; justify-content: space-between; align-items: center; gap: 12px;">
                <div>
                  <div style="font-size: 12px; text-transform: uppercase; letter-spacing: 0.05em; color: #6b7280;">
                    Environment Preflight (rsi doctor --timeout-sec 2)
                  </div>
                  <div style="font-size: 16px; font-weight: 700; color: {status_color}; margin-top: 2px;">
                    {html.escape(status_label)}
                  </div>
                </div>
                <div style="font-size: 13px; color: #374151;">
                  <strong>{ok_count}</strong> OK &middot;
                  <strong>{skip_count}</strong> optional skipped &middot;
                  <strong>{fail_count}</strong> failed &middot;
                  binary: <code>{html.escape(RSI_BIN_LABEL)}</code>
                </div>
              </div>
            </div>
            """
        )
    )


Repository layout : $REPO_ROOT (config: examples/ai-newsletter-eval/rsi.yaml)
Resolved rsi CLI  : target/release/rsi



--- Live `rsi doctor --timeout-sec 2` ---
RSI Doctor: READY
Recommended auto driver: agy_cli

  [OK]   runtime:git            git version 2.39.5 (Apple Git-154)
  [OK]   runtime:python3        Python 3.14.9
  [OK]   runtime:node           v22.16.0
  [OK]   runtime:pnpm           10.33.1
  [SKIP] driver:command         RSI_AGENT_CMD is not set
  [OK]   driver:agy_cli         1.5.18
  [OK]   driver:claude_cli      2.1.252 (Claude Code)
  [SKIP] driver:codex_cli       `codex` not found on PATH
  [SKIP] driver:gemini_cli      `gemini` not found on PATH
  [SKIP] driver:anthropic_api   ANTHROPIC_API_KEY not set
  [SKIP] driver:gemini_api      GEMINI_API_KEY / GOOGLE_API_KEY not set
  [SKIP] driver:openai_api      OPENAI_API_KEY not set
  [OK]   publish:github         `gh` CLI is authenticated
  [OK]   config                 valid config `ai-engineer-newsletter-model-releases` ($REPO_ROOT/examples/ai-newsletter-eval/rsi.yaml)


## 3. Interactive Configuration & Dataset Explorer

Next, we load `examples/ai-newsletter-eval/rsi.yaml` and parse `examples/ai-newsletter-eval/inputs.jsonl` to inspect the 4 Golden-Master workloads, their payload schemas, and which cases trigger the `openRouterCount >= limit` (`remainingSlots === 0`) fast-path regime.

Then we expose an interactive parameter panel (`ipywidgets` in JupyterLab/VS Code, with an automatic plain-Python fallback for headless and static GitHub views) that feeds directly into the command builders in Sections 4, 5, and 8.


In [2]:
import yaml

with CONFIG_PATH.open("r", encoding="utf-8") as fh:
    RSI_CONFIG = yaml.safe_load(fh)

case_rows = []
all_providers: set[str] = set()
field_names: list[str] = []
total_bytes = 0
total_openrouter = 0
total_docs = 0

with INPUTS_PATH.open("r", encoding="utf-8") as fh:
    for idx, raw_line in enumerate(fh):
        stripped = raw_line.strip()
        if not stripped:
            continue
        line_bytes = len(raw_line.encode("utf-8"))
        total_bytes += line_bytes
        payload = json.loads(stripped)
        openrouter_items, doc_items, opts = payload["args"]
        limit = opts.get("limit")
        official_docs = sum(1 for item in doc_items if item.get("announcementUrl"))
        providers = sorted(
            {
                item["providerId"]
                for item in (*openrouter_items, *doc_items)
                if "providerId" in item
            }
        )
        all_providers.update(providers)
        if not field_names and openrouter_items:
            field_names = list(openrouter_items[0].keys())

        total_openrouter += len(openrouter_items)
        total_docs += len(doc_items)

        if len(openrouter_items) == 0 and len(doc_items) == 0:
            regime = "Empty edge case"
        elif limit is not None and len(openrouter_items) >= limit:
            regime = "Scale workload (openRouter >= limit -> remainingSlots == 0)"
        else:
            regime = "Enrichment + dedupe smoke test (remainingSlots > 0)"

        case_rows.append(
            {
                "case": idx,
                "bytes": line_bytes,
                "openrouter": len(openrouter_items),
                "docs": len(doc_items),
                "official_docs": official_docs,
                "providers": len(providers),
                "limit": limit,
                "regime": regime,
            }
        )

print(
    f"Loaded {rel_display(CONFIG_PATH)} -> eval '{RSI_CONFIG['name']}' "
    f"({RSI_CONFIG['language']}, wrapper={RSI_CONFIG['eval']['wrapper']['kind']})"
)
print(
    f"Loaded {rel_display(INPUTS_PATH)} -> {len(case_rows)} cases, "
    f"{total_bytes:,} bytes ({total_bytes / 1024:.1f} KiB), "
    f"{total_openrouter + total_docs} WatchedModelRelease objects "
    f"({total_openrouter} OpenRouter + {total_docs} announcements)"
)
print(f"Schema fields ({len(field_names)}): {', '.join(field_names)}")
print(f"Providers ({len(all_providers)}): {', '.join(sorted(all_providers))}")

tr_html = "".join(
    f"""
    <tr style="border-top: 1px solid #e5e7eb;">
      <td style="padding: 8px 10px; font-weight: 600;">Case {r['case']}</td>
      <td style="padding: 8px 10px; text-align: right; font-variant-numeric: tabular-nums;">{r['bytes']:,} B</td>
      <td style="padding: 8px 10px; text-align: right;">{r['openrouter']}</td>
      <td style="padding: 8px 10px; text-align: right;">{r['docs']} ({r['official_docs']} official)</td>
      <td style="padding: 8px 10px; text-align: right;">{r['providers']}</td>
      <td style="padding: 8px 10px; text-align: right;">{r['limit']}</td>
      <td style="padding: 8px 10px; color: #374151;">{html.escape(r['regime'])}</td>
    </tr>
    """
    for r in case_rows
)

blockers_list = ", ".join(b["id"] for b in RSI_CONFIG["metrics"]["blockers"])
objectives_list = ", ".join(
    f"{o['id']} ({o['direction']}, w={o['weight']})"
    for o in RSI_CONFIG["metrics"]["objectives"]
)

display(
    HTML(
        f"""
        <div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
                    background: #ffffff; color: #1f2937; border: 1px solid #e5e7eb;
                    border-radius: 10px; padding: 14px 16px; margin: 10px 0;">
          <div style="font-size: 14px; font-weight: 700; margin-bottom: 6px;">
            Golden-Master Fixture Summary (<code>examples/ai-newsletter-eval/inputs.jsonl</code>)
          </div>
          <div style="font-size: 12.5px; color: #4b5563; margin-bottom: 10px;">
            <strong>Target:</strong> <code>{html.escape(RSI_CONFIG['eval']['wrapper']['target'])}</code> &middot;
            <strong>Blockers:</strong> <code>golden_equivalence</code>, <code>{html.escape(blockers_list)}</code> &middot;
            <strong>Objectives:</strong> {html.escape(objectives_list)}<br/>
            <strong>Date hydration:</strong> ISO timestamps encoded as <code>{{"$date": "..."}}</code> are hydrated into JS <code>Date</code> objects by the <code>ts_fn</code> runner.
          </div>
          <table style="width: 100%; border-collapse: collapse; font-size: 12.5px;">
            <thead>
              <tr style="background: #f9fafb; color: #374151; text-align: left;">
                <th style="padding: 8px 10px;">Case</th>
                <th style="padding: 8px 10px; text-align: right;">Payload</th>
                <th style="padding: 8px 10px; text-align: right;">OpenRouter</th>
                <th style="padding: 8px 10px; text-align: right;">Announcements</th>
                <th style="padding: 8px 10px; text-align: right;">Providers</th>
                <th style="padding: 8px 10px; text-align: right;">Limit</th>
                <th style="padding: 8px 10px;">Regime</th>
              </tr>
            </thead>
            <tbody>{tr_html}</tbody>
          </table>
        </div>
        """
    )
)


Loaded examples/ai-newsletter-eval/rsi.yaml -> eval 'ai-engineer-newsletter-model-releases' (typescript, wrapper=ts_fn)
Loaded examples/ai-newsletter-eval/inputs.jsonl -> 4 cases, 262,987 bytes (256.8 KiB), 586 WatchedModelRelease objects (203 OpenRouter + 383 announcements)
Schema fields (14): key, providerId, providerLabel, modelName, modelId, url, releasedAt, origin, sourceLabel, contextLabel, pricingLabel, announcementUrl, announcementLabel, isNew
Providers (9): alibaba, anthropic, deepseek, google, mistral, nvidia, openai, tencent, xai


Case,Payload,OpenRouter,Announcements,Providers,Limit,Regime
Case 0,34 B,0,0 (0 official),0,15,Empty edge case
Case 1,"2,733 B",3,3 (2 official),4,15,Enrichment + dedupe smoke test (remainingSlots > 0)
Case 2,"79,819 B",60,120 (60 official),9,20,Scale workload (openRouter >= limit -> remainingSlots == 0)
Case 3,"180,401 B",140,260 (173 official),9,25,Scale workload (openRouter >= limit -> remainingSlots == 0)


In [3]:
# Interactive run configuration panel (ipywidgets when available, clean dict fallback otherwise)
_default_driver = str(RSI_CONFIG.get("agent", {}).get("driver", "agy_cli")).replace("_", "-")
_default_publish = str(RSI_CONFIG.get("repo", {}).get("publish_on_complete", "bundle_only")).replace("_", "-")

RUN_PARAMS: dict[str, object] = {
    "target_ratio": float(RSI_CONFIG["metrics"]["objectives"][0].get("target_ratio", 2.0)),
    "paired_ab_trials": int(RSI_CONFIG["metrics"]["objectives"][0].get("trials", 3)),
    "max_repair_turns": 2,
    "max_steps": int(RSI_CONFIG.get("evolution", {}).get("max_steps", 3)),
    "driver": _default_driver,
    "publish": _default_publish,  # pull-request | zero-pr | bundle-only | branch | tag
    "auto_merge": False,
    "base_branch": str(RSI_CONFIG.get("repo", {}).get("ref", "main")),
}


def build_evolve_command(
    params: dict[str, object],
    repo_override: str | None = None,
    dry_run: bool = False,
    rsi_cmd: str = "rsi",
    state_dir: str | None = None,
) -> list[str]:
    """Construct a valid `rsi evolve` command matching src/cli.rs flags."""
    cmd = [rsi_cmd, "evolve", "--config", "examples/ai-newsletter-eval/rsi.yaml"]
    if repo_override:
        cmd.extend(["--repo", repo_override])
    if dry_run:
        cmd.append("--dry-run")
        return cmd

    cmd.extend(
        [
            "--driver",
            str(params["driver"]),
            "--target-ratio",
            f"{float(params['target_ratio']):.2f}",
            "--paired-ab-trials",
            str(int(params["paired_ab_trials"])),
            "--max-repair-turns",
            str(int(params["max_repair_turns"])),
            "--max-steps",
            str(int(params["max_steps"])),
            "--publish",
            str(params["publish"]),
        ]
    )
    if params["publish"] == "pull-request" and params.get("auto_merge"):
        cmd.append("--auto-merge")
    if params["publish"] in ("pull-request", "zero-pr") and params.get("base_branch"):
        cmd.extend(["--base-branch", str(params["base_branch"])])
    if state_dir:
        cmd.extend(["--state-dir", state_dir])
    return cmd


def render_params_summary(params: dict[str, object], mode_label: str) -> HTML:
    preview = shlex.join(build_evolve_command(params, repo_override="./AIEngineerNewsLetter"))
    rows = "".join(
        f"""
        <tr style="border-top: 1px solid #e5e7eb;">
          <td style="padding: 6px 10px; font-weight: 600; color: #374151;"><code>{html.escape(k)}</code></td>
          <td style="padding: 6px 10px; color: #111827;"><code>{html.escape(str(v))}</code></td>
        </tr>
        """
        for k, v in params.items()
    )
    return HTML(
        f"""
        <div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
                    background: #ffffff; color: #1f2937; border: 1px solid #e5e7eb;
                    border-radius: 10px; padding: 14px 16px; margin: 8px 0;">
          <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 8px;">
            <span style="font-size: 14px; font-weight: 700;">Active Evolution Parameters (<code>RUN_PARAMS</code>)</span>
            <span style="font-size: 11.5px; background: #eff6ff; color: #1d4ed8; padding: 2px 8px; border-radius: 999px;">
              {html.escape(mode_label)}
            </span>
          </div>
          <table style="width: 100%; border-collapse: collapse; font-size: 12.5px; margin-bottom: 10px;">
            <tbody>{rows}</tbody>
          </table>
          <div style="font-size: 12px; color: #4b5563; margin-bottom: 4px;">Generated CLI Command Preview:</div>
          <pre style="margin: 0; padding: 10px 12px; background: #0f172a; color: #e2e8f0;
                      border-radius: 6px; font-size: 12px; overflow-x: auto;">{html.escape(preview)}</pre>
        </div>
        """
    )


try:
    import ipywidgets as widgets

    w_target_ratio = widgets.FloatSlider(
        value=float(RUN_PARAMS["target_ratio"]),
        min=1.2,
        max=10.0,
        step=0.2,
        description="target_ratio:",
        style={"description_width": "135px"},
        layout=widgets.Layout(width="420px"),
    )
    w_paired_ab = widgets.IntSlider(
        value=int(RUN_PARAMS["paired_ab_trials"]),
        min=1,
        max=9,
        step=1,
        description="paired_ab_trials:",
        style={"description_width": "135px"},
        layout=widgets.Layout(width="420px"),
    )
    w_repair = widgets.IntSlider(
        value=int(RUN_PARAMS["max_repair_turns"]),
        min=0,
        max=5,
        step=1,
        description="max_repair_turns:",
        style={"description_width": "135px"},
        layout=widgets.Layout(width="420px"),
    )
    w_max_steps = widgets.IntSlider(
        value=int(RUN_PARAMS["max_steps"]),
        min=1,
        max=10,
        step=1,
        description="max_steps:",
        style={"description_width": "135px"},
        layout=widgets.Layout(width="420px"),
    )
    w_driver = widgets.Dropdown(
        options=[
            "agy-cli",
            "auto",
            "claude-cli",
            "codex-cli",
            "gemini-cli",
            "anthropic-api",
            "openai-api",
            "gemini-api",
            "command",
        ],
        value=str(RUN_PARAMS["driver"]),
        description="driver:",
        style={"description_width": "135px"},
        layout=widgets.Layout(width="340px"),
    )
    w_publish = widgets.Dropdown(
        options=["pull-request", "zero-pr", "bundle-only", "branch", "tag"],
        value=str(RUN_PARAMS["publish"]),
        description="publish mode:",
        style={"description_width": "135px"},
        layout=widgets.Layout(width="340px"),
    )
    w_auto_merge = widgets.Checkbox(
        value=bool(RUN_PARAMS["auto_merge"]),
        description="auto_merge (pull-request mode)",
        indent=False,
    )
    w_base_branch = widgets.Text(
        value=str(RUN_PARAMS["base_branch"]),
        description="base_branch:",
        style={"description_width": "135px"},
        layout=widgets.Layout(width="340px"),
    )
    cmd_out = widgets.Output()

    def _sync_params(_change=None):
        RUN_PARAMS.update(
            {
                "target_ratio": round(float(w_target_ratio.value), 2),
                "paired_ab_trials": int(w_paired_ab.value),
                "max_repair_turns": int(w_repair.value),
                "max_steps": int(w_max_steps.value),
                "driver": str(w_driver.value),
                "publish": str(w_publish.value),
                "auto_merge": bool(w_auto_merge.value),
                "base_branch": str(w_base_branch.value).strip() or "main",
            }
        )
        with cmd_out:
            cmd_out.clear_output(wait=True)
            display(render_params_summary(RUN_PARAMS, "ipywidgets interactive panel"))

    for widget in (
        w_target_ratio,
        w_paired_ab,
        w_repair,
        w_max_steps,
        w_driver,
        w_publish,
        w_auto_merge,
        w_base_branch,
    ):
        widget.observe(_sync_params, names="value")

    panel = widgets.VBox(
        [
            widgets.HBox([widgets.VBox([w_target_ratio, w_paired_ab, w_repair, w_max_steps]),
                          widgets.VBox([w_driver, w_publish, w_base_branch, w_auto_merge])]),
            cmd_out,
        ]
    )
    _sync_params()
    display(panel)
except ImportError:
    print("ipywidgets not installed -> using plain-Python RUN_PARAMS dict (edit RUN_PARAMS above to customize).")
    display(render_params_summary(RUN_PARAMS, "plain-Python config fallback"))


ipywidgets not installed -> using plain-Python RUN_PARAMS dict (edit RUN_PARAMS above to customize).


target_ratio,2.0
paired_ab_trials,3
max_repair_turns,2
max_steps,3
driver,agy-cli
publish,bundle-only
auto_merge,False
base_branch,main


## 4. Zero-LLM Preflight Dry Run (`rsi evolve --dry-run`)

Before spending any coding-agent turns or API tokens, `rsi evolve --dry-run` validates the entire evaluation harness inside an ephemeral sandbox (`TempDir`, auto-cleaned with zero state mutations):

1. Snapshots `protected_paths` (**61 files**) and verifies the target repository builds/imports cleanly.
2. Captures the **Step-0 Golden-Master** output table via the `ts_fn` wrapper.
3. Executes all blockers (`golden_equivalence`, `shared_unit_tests`, `shared_typecheck`).
4. Benchmarks Step-0 baseline objectives and computes the **IQR noise-floor calibration** so future promotions must clear real jitter.

> **Portable TypeScript Monorepo Note:** `AIEngineerNewsLetter` uses extensionless TypeScript imports (`./prompts`) and `pnpm` scripts that rely on installed `node_modules` (`tsx`, `typescript`). When cloning from the remote HTTPS URL alone, no local `node_modules` exist to symlink into the sandbox. Clone once locally, run `pnpm install`, and pass `--repo ./AIEngineerNewsLetter` (or set `AI_NEWSLETTER_REPO`).


In [4]:
# Flip to True after cloning AIEngineerNewsLetter locally and running `pnpm install`.
RUN_DRY_RUN = False
LOCAL_NEWSLETTER_REPO = os.environ.get("AI_NEWSLETTER_REPO", "")

portable_repo_arg = LOCAL_NEWSLETTER_REPO or "./AIEngineerNewsLetter"
dry_run_cmd = build_evolve_command(
    RUN_PARAMS,
    repo_override=portable_repo_arg,
    dry_run=True,
    rsi_cmd=RSI_BIN_LABEL if RSI_BIN else "rsi",
)

print("Portable one-time local checkout + zero-LLM dry run:")
print("  git clone https://github.com/DontAskFuture/AIEngineerNewsLetter.git ./AIEngineerNewsLetter")
print("  pnpm --dir ./AIEngineerNewsLetter install")
print(f"  {shlex.join(build_evolve_command(RUN_PARAMS, repo_override='./AIEngineerNewsLetter', dry_run=True))}")

VERIFIED_DRY_RUN_REFERENCE = """RSI Dry Run: ai-engineer-newsletter-model-releases (./AIEngineerNewsLetter)
Base SHA: 3724cb21 | Language: typescript | Resolved Driver: agy_cli | Protected files: 61
Blockers: PASSED (3 checked)
  - golden_equivalence [pass] (163ms)
  - shared_unit_tests [pass] (612ms)
  - shared_typecheck [pass] (639ms)
Baseline Objectives & Noise Calibration:
  - ops_per_sec (Maximize): baseline=137.9000, target=275.8000, min_delta=11.0320 -> calibrated=11.0320 | IQR=0.1500
  - p95_ms (Minimize): baseline=7.9600, target=-, min_delta=0.3980 -> calibrated=0.3980 | IQR=0.2300"""

if RUN_DRY_RUN:
    if not RSI_BIN:
        raise RuntimeError("rsi binary not found; build with `cargo build --release` first.")
    if not LOCAL_NEWSLETTER_REPO or not Path(LOCAL_NEWSLETTER_REPO).expanduser().is_dir():
        raise FileNotFoundError(
            "Set LOCAL_NEWSLETTER_REPO (or AI_NEWSLETTER_REPO env var) to a local clone of "
            "AIEngineerNewsLetter with `pnpm install` completed."
        )
    live_cmd = build_evolve_command(
        RUN_PARAMS,
        repo_override=str(Path(LOCAL_NEWSLETTER_REPO).expanduser().resolve()),
        dry_run=True,
        rsi_cmd=RSI_BIN,
    )
    proc = subprocess.run(
        live_cmd,
        cwd=REPO_ROOT,
        stdin=subprocess.DEVNULL,
        capture_output=True,
        text=True,
        check=False,
    )
    print(f"\n--- Live Dry-Run Output (exit {proc.returncode}) ---")
    print(sanitize_text(proc.stdout.strip()))
    if proc.stderr.strip():
        print("\n[stderr]\n" + sanitize_text(proc.stderr.strip()))
else:
    print("\nRUN_DRY_RUN = False -> showing verified sanitized dry-run output (~5s on local clone):")
    print(VERIFIED_DRY_RUN_REFERENCE)


Portable one-time local checkout + zero-LLM dry run:
  git clone https://github.com/DontAskFuture/AIEngineerNewsLetter.git ./AIEngineerNewsLetter
  pnpm --dir ./AIEngineerNewsLetter install
  rsi evolve --config examples/ai-newsletter-eval/rsi.yaml --repo ./AIEngineerNewsLetter --dry-run

RUN_DRY_RUN = False -> showing verified sanitized dry-run output (~5s on local clone):
RSI Dry Run: ai-engineer-newsletter-model-releases (./AIEngineerNewsLetter)
Base SHA: 3724cb21 | Language: typescript | Resolved Driver: agy_cli | Protected files: 61
Blockers: PASSED (3 checked)
  - golden_equivalence [pass] (163ms)
  - shared_unit_tests [pass] (612ms)
  - shared_typecheck [pass] (639ms)
Baseline Objectives & Noise Calibration:
  - ops_per_sec (Maximize): baseline=137.9000, target=275.8000, min_delta=11.0320 -> calibrated=11.0320 | IQR=0.1500
  - p95_ms (Minimize): baseline=7.9600, target=-, min_delta=0.3980 -> calibrated=0.3980 | IQR=0.2300


## 5. Evolution Runner & Recorded Reference Replay

By default (`RUN_LIVE_EVOLUTION = False`), this cell replays the real sanitized ledger data from **`run_20261123_042237_1dd900`** so any reader can inspect the exact metrics, trial spreads, and diff walkthrough instantaneously without spending API credits or agent quota.

Flip `RUN_LIVE_EVOLUTION = True` to launch `rsi evolve` live with your `RUN_PARAMS` settings from Section 3.


In [5]:
# Flip to True to run live autonomous evolution with your configured driver in RUN_PARAMS.
RUN_LIVE_EVOLUTION = False

RECORDED_RUN = {
    "source": "recorded_reference_replay",
    "run_id": "run_20261123_042237_1dd900",
    "name": "ai-engineer-newsletter-model-releases",
    "repo_url": "https://github.com/DontAskFuture/AIEngineerNewsLetter.git",
    "ref": "main",
    "language": "typescript",
    "target": "packages/shared/src/model-releases.ts:mergeWatchedModelReleases",
    "status": "GOAL_REACHED",
    "spec_revision_id": "spec_b4835924",
    "started_at": "2026-11-23T04:22:41Z",
    "finished_at": "2026-11-23T04:34:30Z",
    "steps_completed": 1,
    "max_steps": 3,
    "promoted_steps": 1,
    "reverted_steps": 0,
    "repair_turns": 0,
    "step_1_duration_ms": 708765,
    "protected_files_count": 61,
    "base_sha": "3724cb21",
    "best_sha": "8c25758d",
    "base_ref": "refs/rsi/run_20261123_042237_1dd900/step-0",
    "best_ref": "refs/rsi/run_20261123_042237_1dd900/best",
    "diff_stats": {
        "files_changed": 1,
        "insertions": 1053,
        "deletions": 109,
        "touched_files": ["packages/shared/src/model-releases.ts"],
    },
    "hypothesis": (
        "Pre-indexing OpenRouter models by provider and rarest anchor token with a "
        "streaming ASCII bitmask filter, bounded top-k heap selection with early exit "
        "when OpenRouter releases fill the limit, and per-provider inverted-index "
        "announcement deduplication."
    ),
    "objectives": {
        "ops_per_sec": {
            "direction": "MAXIMIZE",
            "unit": "ops/sec",
            "weight": 0.7,
            "target_ratio": 2.0,
            "baseline": 141.1787,
            "target": 282.3574,
            "evolved_median": 6443.0015,
            "step_1_trials": [6562.877, 6125.9958, 6443.0015],
        },
        "p95_ms": {
            "direction": "MINIMIZE",
            "unit": "ms",
            "weight": 0.3,
            "baseline": 7.587291,
            "evolved_median": 0.192458,
            "step_1_trials": [0.192458, 0.256416, 0.186709],
        },
    },
    "blockers": [
        {
            "id": "golden_equivalence",
            "command": "ts_fn exact_output Golden-Master comparison (4 cases)",
            "passed": True,
            "duration_ms": 148,
        },
        {
            "id": "shared_unit_tests",
            "command": "pnpm --dir packages/shared test",
            "passed": True,
            "duration_ms": 488,
        },
        {
            "id": "shared_typecheck",
            "command": "pnpm --dir packages/shared typecheck",
            "passed": True,
            "duration_ms": 542,
        },
    ],
}

live_preview_cmd = build_evolve_command(
    RUN_PARAMS,
    repo_override=LOCAL_NEWSLETTER_REPO or "./AIEngineerNewsLetter",
)
print("Live evolution command from Section 3 RUN_PARAMS:")
print(f"  {shlex.join(live_preview_cmd)}\n")

if RUN_LIVE_EVOLUTION:
    if not RSI_BIN:
        raise RuntimeError("rsi binary not found; build with `cargo build --release` first.")
    target_repo = (
        str(Path(LOCAL_NEWSLETTER_REPO).expanduser().resolve())
        if LOCAL_NEWSLETTER_REPO
        else RSI_CONFIG["repo"]["url"]
    )
    live_state_dir = REPO_ROOT / ".rsi-notebook-state"
    cmd = build_evolve_command(
        RUN_PARAMS,
        repo_override=target_repo,
        rsi_cmd=RSI_BIN,
        state_dir=str(live_state_dir),
    )
    proc = subprocess.run(
        cmd,
        cwd=REPO_ROOT,
        stdin=subprocess.DEVNULL,
        capture_output=True,
        text=True,
        check=False,
    )
    print(f"--- Live `rsi evolve` output (exit {proc.returncode}) ---")
    print(sanitize_text(proc.stdout.strip()))
    if proc.stderr.strip():
        print("\n[stderr]\n" + sanitize_text(proc.stderr.strip()))
    if proc.returncode != 0:
        raise RuntimeError("Live evolution failed; inspect sanitized output above.")

    status_proc = subprocess.run(
        [RSI_BIN, "status", "--json", "--state-dir", str(live_state_dir)],
        stdin=subprocess.DEVNULL,
        capture_output=True,
        text=True,
        check=True,
    )
    raw_status = json.loads(status_proc.stdout)
    run_obj = raw_status["run"]
    steps_list = raw_status.get("steps", [])
    promoted = [s for s in steps_list if str(s.get("outcome", "")).lower() == "promoted"]
    last_promoted = promoted[-1] if promoted else (steps_list[-1] if steps_list else {})
    meas_by_id = {m["metric_id"]: m for m in last_promoted.get("objective_measurements", [])}

    ACTIVE_RUN = dict(RECORDED_RUN)
    ACTIVE_RUN["source"] = "live_run"
    ACTIVE_RUN["run_id"] = run_obj["id"]
    ACTIVE_RUN["status"] = run_obj["status"]
    ACTIVE_RUN["steps_completed"] = run_obj["current_step"]
    ACTIVE_RUN["max_steps"] = run_obj["max_steps"]
    ACTIVE_RUN["promoted_steps"] = len(promoted)
    ACTIVE_RUN["reverted_steps"] = len(steps_list) - len(promoted)
    ACTIVE_RUN["base_sha"] = str(run_obj.get("base_commit_sha", ""))[:8]
    ACTIVE_RUN["best_sha"] = str(run_obj.get("best_commit_sha", ""))[:8]
    if "ops_per_sec" in run_obj.get("baseline_scorecard", {}):
        base_ops = float(run_obj["baseline_scorecard"]["ops_per_sec"])
        best_ops = float(run_obj["best_scorecard"].get("ops_per_sec", base_ops))
        ACTIVE_RUN["objectives"]["ops_per_sec"] = {
            **RECORDED_RUN["objectives"]["ops_per_sec"],
            "baseline": base_ops,
            "target": base_ops * float(RUN_PARAMS["target_ratio"]),
            "evolved_median": best_ops,
            "step_1_trials": meas_by_id.get("ops_per_sec", {}).get("trial_values", [best_ops]),
        }
    if "p95_ms" in run_obj.get("baseline_scorecard", {}):
        base_p95 = float(run_obj["baseline_scorecard"]["p95_ms"])
        best_p95 = float(run_obj["best_scorecard"].get("p95_ms", base_p95))
        ACTIVE_RUN["objectives"]["p95_ms"] = {
            **RECORDED_RUN["objectives"]["p95_ms"],
            "baseline": base_p95,
            "evolved_median": best_p95,
            "step_1_trials": meas_by_id.get("p95_ms", {}).get("trial_values", [best_p95]),
        }
else:
    ACTIVE_RUN = RECORDED_RUN
    ops = ACTIVE_RUN["objectives"]["ops_per_sec"]
    p95 = ACTIVE_RUN["objectives"]["p95_ms"]
    ops_speedup = ops["evolved_median"] / ops["baseline"]
    p95_reduction = p95["baseline"] / p95["evolved_median"]
    blocker_total = sum(b["duration_ms"] for b in ACTIVE_RUN["blockers"])
    step_sec = ACTIVE_RUN["step_1_duration_ms"] / 1000.0

    print("RUN_LIVE_EVOLUTION = False -> replaying real sanitized ledger run:")
    print(f"  Run      : {ACTIVE_RUN['run_id']} ({ACTIVE_RUN['name']})")
    print(
        f"  Status   : {ACTIVE_RUN['status']} | Step {ACTIVE_RUN['steps_completed']}/{ACTIVE_RUN['max_steps']} "
        f"(promoted: {ACTIVE_RUN['promoted_steps']}, reverted: {ACTIVE_RUN['reverted_steps']}, "
        f"repair turns: {ACTIVE_RUN['repair_turns']})"
    )
    print(
        f"  Commits  : {ACTIVE_RUN['base_sha']} -> {ACTIVE_RUN['best_sha']} "
        f"({ACTIVE_RUN['diff_stats']['files_changed']} file changed, "
        f"+{ACTIVE_RUN['diff_stats']['insertions']} / -{ACTIVE_RUN['diff_stats']['deletions']}, "
        f"{ACTIVE_RUN['protected_files_count']} protected files untouched)"
    )
    print(f"  Duration : {ACTIVE_RUN['step_1_duration_ms']:,} ms ({step_sec:.1f}s / 11m 48.8s)")
    print("  Metrics  :")
    print(
        f"    - ops_per_sec : {ops['baseline']:.4f} -> {ops['evolved_median']:.4f} "
        f"({ops_speedup:.2f}x speedup, +{(ops_speedup - 1) * 100:.1f}%, trials={ops['step_1_trials']})"
    )
    print(
        f"    - p95_ms      : {p95['baseline']:.4f} ms -> {p95['evolved_median']:.4f} ms "
        f"({p95_reduction:.2f}x lower, -{(1 - p95['evolved_median'] / p95['baseline']) * 100:.2f}%, "
        f"trials={p95['step_1_trials']})"
    )
    print(
        f"  Blockers : {len(ACTIVE_RUN['blockers'])}/{len(ACTIVE_RUN['blockers'])} PASSED "
        f"(total {blocker_total:,} ms)"
    )


Live evolution command from Section 3 RUN_PARAMS:
  rsi evolve --config examples/ai-newsletter-eval/rsi.yaml --repo ./AIEngineerNewsLetter --driver agy-cli --target-ratio 2.00 --paired-ab-trials 3 --max-repair-turns 2 --max-steps 3 --publish bundle-only

RUN_LIVE_EVOLUTION = False -> replaying real sanitized ledger run:
  Run      : run_20261123_042237_1dd900 (ai-engineer-newsletter-model-releases)
  Status   : GOAL_REACHED | Step 1/3 (promoted: 1, reverted: 0, repair turns: 0)
  Commits  : 3724cb21 -> 8c25758d (1 file changed, +1053 / -109, 61 protected files untouched)
  Duration : 708,765 ms (708.8s / 11m 48.8s)
  Metrics  :
    - ops_per_sec : 141.1787 -> 6443.0015 (45.64x speedup, +4463.7%, trials=[6562.877, 6125.9958, 6443.0015])
    - p95_ms      : 7.5873 ms -> 0.1925 ms (39.42x lower, -97.46%, trials=[0.192458, 0.256416, 0.186709])
  Blockers : 3/3 PASSED (total 1,178 ms)


## 6. Interactive Visual Scorecard & Charts

All charts below are rendered using pure Python and inline **SVG** inside `IPython.display.HTML`—zero external plotting libraries (`matplotlib`/`plotly`) required—so they render crisply in JupyterLab, VS Code, and GitHub's native notebook viewer.


In [6]:
import xml.etree.ElementTree as ET

ops = ACTIVE_RUN["objectives"]["ops_per_sec"]
p95 = ACTIVE_RUN["objectives"]["p95_ms"]
blockers = ACTIVE_RUN["blockers"]

ops_speedup = ops["evolved_median"] / ops["baseline"]
ops_pct = (ops_speedup - 1.0) * 100.0
p95_factor = p95["baseline"] / p95["evolved_median"]
p95_drop_pct = (1.0 - p95["evolved_median"] / p95["baseline"]) * 100.0
blocker_total_ms = sum(b["duration_ms"] for b in blockers)


def build_comparison_svg() -> str:
    """Side-by-side SVG chart for Baseline vs Target vs Evolved throughput & p95 latency."""
    bar_max_w = 260.0
    ops_max = max(ops["baseline"], ops["target"], ops["evolved_median"])
    p95_max = max(p95["baseline"], p95["evolved_median"])

    ops_items = [
        ("Baseline (Step 0)", ops["baseline"], "#94a3b8", f"{ops['baseline']:,.2f}"),
        ("2.0x Goal Target", ops["target"], "#f59e0b", f"{ops['target']:,.2f}"),
        ("Evolved (Step 1)", ops["evolved_median"], "#16a34a", f"{ops['evolved_median']:,.2f}"),
    ]
    p95_items = [
        ("Baseline (Step 0)", p95["baseline"], "#ef4444", f"{p95['baseline']:.2f} ms"),
        ("Evolved (Step 1)", p95["evolved_median"], "#0ea5e9", f"{p95['evolved_median']:.2f} ms"),
    ]

    parts = [
        '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 920 265" width="920" height="265" '
        'role="img" aria-label="Baseline vs Evolved Throughput and p95 Latency">',
        '<rect x="0" y="0" width="920" height="265" rx="12" fill="#ffffff" stroke="#e5e7eb"/>',
        # Left panel: Throughput
        '<text x="24" y="34" font-family="-apple-system,Segoe UI,sans-serif" font-size="15" font-weight="700" fill="#111827">'
        'Throughput (ops/sec) — Higher is Better</text>',
        '<rect x="336" y="16" width="102" height="24" rx="12" fill="#dcfce7"/>',
        f'<text x="387" y="32" text-anchor="middle" font-family="-apple-system,Segoe UI,sans-serif" '
        f'font-size="12" font-weight="700" fill="#15803d">{ops_speedup:.2f}x faster</text>',
    ]

    y = 74
    for label, val, color, val_label in ops_items:
        w = max(8.0, (val / ops_max) * bar_max_w)
        parts.append(
            f'<text x="24" y="{y + 15}" font-family="-apple-system,Segoe UI,sans-serif" '
            f'font-size="12.5" fill="#374151">{html.escape(label)}</text>'
        )
        parts.append(f'<rect x="148" y="{y}" width="{bar_max_w:.1f}" height="22" rx="5" fill="#f3f4f6"/>')
        parts.append(f'<rect x="148" y="{y}" width="{w:.1f}" height="22" rx="5" fill="{color}"/>')
        label_x = 148 + w + 8 if w < 185 else 148 + w - 8
        anchor = "start" if w < 185 else "end"
        txt_fill = "#111827" if w < 185 else "#ffffff"
        parts.append(
            f'<text x="{label_x:.1f}" y="{y + 15}" text-anchor="{anchor}" '
            f'font-family="-apple-system,Segoe UI,sans-serif" font-size="12" font-weight="700" fill="{txt_fill}">'
            f'{html.escape(val_label)}</text>'
        )
        y += 52

    parts.append(
        f'<text x="24" y="242" font-family="-apple-system,Segoe UI,sans-serif" font-size="11.5" fill="#6b7280">'
        f'Delta: +{ops["evolved_median"] - ops["baseline"]:,.2f} ops/sec (+{ops_pct:,.1f}% vs. Step 0 baseline)</text>'
    )

    # Divider
    parts.append('<line x1="460" y1="18" x2="460" y2="246" stroke="#e5e7eb" stroke-dasharray="4 4"/>')

    # Right panel: p95 Latency
    parts.append(
        '<text x="484" y="34" font-family="-apple-system,Segoe UI,sans-serif" font-size="15" font-weight="700" fill="#111827">'
        'p95 Latency (ms) — Lower is Better</text>'
    )
    parts.append('<rect x="786" y="16" width="110" height="24" rx="12" fill="#e0f2fe"/>')
    parts.append(
        f'<text x="841" y="32" text-anchor="middle" font-family="-apple-system,Segoe UI,sans-serif" '
        f'font-size="12" font-weight="700" fill="#0369a1">-{p95_drop_pct:.1f}% ({p95_factor:.1f}x)</text>'
    )

    y = 88
    for label, val, color, val_label in p95_items:
        w = max(8.0, (val / p95_max) * bar_max_w)
        parts.append(
            f'<text x="484" y="{y + 17}" font-family="-apple-system,Segoe UI,sans-serif" '
            f'font-size="12.5" fill="#374151">{html.escape(label)}</text>'
        )
        parts.append(f'<rect x="608" y="{y}" width="{bar_max_w:.1f}" height="26" rx="5" fill="#f3f4f6"/>')
        parts.append(f'<rect x="608" y="{y}" width="{w:.1f}" height="26" rx="5" fill="{color}"/>')
        label_x = 608 + w + 8 if w < 185 else 608 + w - 8
        anchor = "start" if w < 185 else "end"
        txt_fill = "#111827" if w < 185 else "#ffffff"
        parts.append(
            f'<text x="{label_x:.1f}" y="{y + 17}" text-anchor="{anchor}" '
            f'font-family="-apple-system,Segoe UI,sans-serif" font-size="12" font-weight="700" fill="{txt_fill}">'
            f'{html.escape(val_label)}</text>'
        )
        y += 66

    parts.append(
        f'<text x="484" y="242" font-family="-apple-system,Segoe UI,sans-serif" font-size="11.5" fill="#6b7280">'
        f'Delta: -{p95["baseline"] - p95["evolved_median"]:.4f} ms ({p95["baseline"]:.4f} ms -> {p95["evolved_median"]:.4f} ms)</text>'
    )
    parts.append("</svg>")
    return "".join(parts)


def build_trials_svg() -> str:
    """Per-trial spread across the 3 Step-1 benchmark trials for ops_per_sec and p95_ms."""
    ops_trials = ops["step_1_trials"]
    p95_trials = p95["step_1_trials"]
    ops_ceil = max(ops_trials) * 1.15
    p95_ceil = max(p95_trials) * 1.25
    chart_h = 118.0
    base_y = 206.0

    parts = [
        '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 920 262" width="920" height="262" '
        'role="img" aria-label="Step 1 Per-Trial Stability Spread">',
        '<rect x="0" y="0" width="920" height="262" rx="12" fill="#ffffff" stroke="#e5e7eb"/>',
        # Left panel header + dedicated median legend pill (no bar-label overlap)
        '<text x="24" y="32" font-family="-apple-system,Segoe UI,sans-serif" font-size="14.5" font-weight="700" fill="#111827">'
        'Step 1 Trial Spread: ops_per_sec</text>',
        '<rect x="294" y="16" width="148" height="24" rx="12" fill="#dcfce7"/>',
        '<line x1="306" y1="28" x2="322" y2="28" stroke="#15803d" stroke-width="2" stroke-dasharray="4 3"/>',
        f'<text x="328" y="32" font-family="-apple-system,Segoe UI,sans-serif" '
        f'font-size="11.5" font-weight="700" fill="#15803d">median {ops["evolved_median"]:,.2f}</text>',
        # Right panel header + dedicated median legend pill (no bar-label overlap)
        '<text x="484" y="32" font-family="-apple-system,Segoe UI,sans-serif" font-size="14.5" font-weight="700" fill="#111827">'
        'Step 1 Trial Spread: p95_ms</text>',
        '<rect x="744" y="16" width="152" height="24" rx="12" fill="#e0f2fe"/>',
        '<line x1="756" y1="28" x2="772" y2="28" stroke="#0369a1" stroke-width="2" stroke-dasharray="4 3"/>',
        f'<text x="778" y="32" font-family="-apple-system,Segoe UI,sans-serif" '
        f'font-size="11.5" font-weight="700" fill="#0369a1">median {p95["evolved_median"]:.3f} ms</text>',
        '<line x1="460" y1="18" x2="460" y2="244" stroke="#e5e7eb" stroke-dasharray="4 4"/>',
    ]

    # Left: ops_per_sec trials (dashed median line behind bars/labels)
    parts.append(f'<line x1="56" y1="{base_y}" x2="424" y2="{base_y}" stroke="#cbd5e1"/>')
    med_ops_y = base_y - (ops["evolved_median"] / ops_ceil) * chart_h
    parts.append(
        f'<line x1="56" y1="{med_ops_y:.1f}" x2="424" y2="{med_ops_y:.1f}" '
        f'stroke="#15803d" stroke-width="1.5" stroke-dasharray="5 4" stroke-opacity="0.8"/>'
    )
    for i, v in enumerate(ops_trials):
        x = 92 + i * 112
        h = (v / ops_ceil) * chart_h
        y_top = base_y - h
        parts.append(f'<rect x="{x}" y="{y_top:.1f}" width="64" height="{h:.1f}" rx="6" fill="#22c55e" fill-opacity="0.85"/>')
        parts.append(
            f'<text x="{x + 32}" y="{y_top - 10:.1f}" text-anchor="middle" font-family="-apple-system,Segoe UI,sans-serif" '
            f'font-size="11.5" font-weight="700" fill="#1f2937">{v:,.1f}</text>'
        )
        parts.append(
            f'<text x="{x + 32}" y="{base_y + 18}" text-anchor="middle" font-family="-apple-system,Segoe UI,sans-serif" '
            f'font-size="11.5" fill="#4b5563">Trial {i + 1}</text>'
        )

    ops_spread_pct = ((max(ops_trials) - min(ops_trials)) / ops["evolved_median"]) * 100.0
    parts.append(
        f'<text x="24" y="246" font-family="-apple-system,Segoe UI,sans-serif" font-size="11.5" fill="#6b7280">'
        f'Range: {min(ops_trials):,.1f} – {max(ops_trials):,.1f} ops/sec (total spread {ops_spread_pct:.1f}% vs. {ops_speedup:.1f}x gain)</text>'
    )

    # Right: p95_ms trials (dashed median line behind bars/labels)
    parts.append(f'<line x1="516" y1="{base_y}" x2="884" y2="{base_y}" stroke="#cbd5e1"/>')
    med_p95_y = base_y - (p95["evolved_median"] / p95_ceil) * chart_h
    parts.append(
        f'<line x1="516" y1="{med_p95_y:.1f}" x2="884" y2="{med_p95_y:.1f}" '
        f'stroke="#0369a1" stroke-width="1.5" stroke-dasharray="5 4" stroke-opacity="0.8"/>'
    )
    for i, v in enumerate(p95_trials):
        x = 552 + i * 112
        h = (v / p95_ceil) * chart_h
        y_top = base_y - h
        parts.append(f'<rect x="{x}" y="{y_top:.1f}" width="64" height="{h:.1f}" rx="6" fill="#38bdf8" fill-opacity="0.9"/>')
        parts.append(
            f'<text x="{x + 32}" y="{y_top - 10:.1f}" text-anchor="middle" font-family="-apple-system,Segoe UI,sans-serif" '
            f'font-size="11.5" font-weight="700" fill="#1f2937">{v:.3f} ms</text>'
        )
        parts.append(
            f'<text x="{x + 32}" y="{base_y + 18}" text-anchor="middle" font-family="-apple-system,Segoe UI,sans-serif" '
            f'font-size="11.5" fill="#4b5563">Trial {i + 1}</text>'
        )

    parts.append(
        f'<text x="484" y="246" font-family="-apple-system,Segoe UI,sans-serif" font-size="11.5" fill="#6b7280">'
        f'Range: {min(p95_trials):.3f} – {max(p95_trials):.3f} ms (all trials &lt; 0.26 ms vs. {p95["baseline"]:.2f} ms baseline)</text>'
    )
    parts.append("</svg>")
    return "".join(parts)


def build_blockers_svg() -> str:
    """Horizontal SVG bar chart of real repository blocker timings."""
    bar_max_w = 480.0
    max_ms = max(b["duration_ms"] for b in blockers)
    colors = ["#6366f1", "#10b981", "#0ea5e9"]

    parts = [
        '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 920 198" width="920" height="198" '
        'role="img" aria-label="Blocker Check Timings">',
        '<rect x="0" y="0" width="920" height="198" rx="12" fill="#ffffff" stroke="#e5e7eb"/>',
        '<text x="24" y="34" font-family="-apple-system,Segoe UI,sans-serif" font-size="15" font-weight="700" fill="#111827">'
        'Correctness &amp; Typecheck Blocker Timings (Step 1 Promotion Gate)</text>',
        '<rect x="752" y="16" width="144" height="24" rx="12" fill="#dcfce7"/>',
        f'<text x="824" y="32" text-anchor="middle" font-family="-apple-system,Segoe UI,sans-serif" '
        f'font-size="12" font-weight="700" fill="#15803d">3/3 PASS &#183; {blocker_total_ms:,} ms</text>',
    ]

    y = 64
    for idx, b in enumerate(blockers):
        w = max(12.0, (b["duration_ms"] / max_ms) * bar_max_w)
        pct = (b["duration_ms"] / blocker_total_ms) * 100.0
        col = colors[idx % len(colors)]
        parts.append(
            f'<text x="24" y="{y + 15}" font-family="ui-monospace,SFMono-Regular,Menlo,monospace" '
            f'font-size="12.5" font-weight="600" fill="#1f2937">{html.escape(b["id"])}</text>'
        )
        parts.append(f'<rect x="206" y="{y}" width="{bar_max_w:.1f}" height="24" rx="6" fill="#f3f4f6"/>')
        parts.append(f'<rect x="206" y="{y}" width="{w:.1f}" height="24" rx="6" fill="{col}"/>')
        parts.append(
            f'<text x="{206 + bar_max_w + 14:.1f}" y="{y + 16}" font-family="-apple-system,Segoe UI,sans-serif" '
            f'font-size="12.5" font-weight="700" fill="#111827">{b["duration_ms"]} ms '
            f'<tspan font-weight="400" fill="#6b7280">({pct:.1f}%)</tspan></text>'
        )
        parts.append('<rect x="836" y="' + str(y + 2) + '" width="56" height="20" rx="10" fill="#dcfce7"/>')
        parts.append(
            f'<text x="864" y="{y + 16}" text-anchor="middle" font-family="-apple-system,Segoe UI,sans-serif" '
            f'font-size="11" font-weight="700" fill="#15803d">PASS</text>'
        )
        y += 42

    parts.append("</svg>")
    return "".join(parts)


svg_comparison = build_comparison_svg()
svg_trials = build_trials_svg()
svg_blockers = build_blockers_svg()

# Validate well-formed SVG XML before display
for svg_name, svg_doc in [
    ("comparison", svg_comparison),
    ("trials", svg_trials),
    ("blockers", svg_blockers),
]:
    ET.fromstring(svg_doc)

scorecard_html = f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; color: #1f2937; max-width: 940px;">
  <div style="display: grid; grid-template-columns: repeat(4, minmax(170px, 1fr)); gap: 10px; margin-bottom: 14px;">
    <div style="background: #ffffff; border: 1px solid #e5e7eb; border-top: 3px solid #16a34a; border-radius: 10px; padding: 12px 14px;">
      <div style="font-size: 11.5px; color: #6b7280; text-transform: uppercase;">Throughput</div>
      <div style="font-size: 22px; font-weight: 800; color: #15803d; margin: 2px 0;">{ops_speedup:.2f}x</div>
      <div style="font-size: 12px; color: #374151;">{ops['baseline']:.2f} &rarr; {ops['evolved_median']:,.2f} ops/s</div>
    </div>
    <div style="background: #ffffff; border: 1px solid #e5e7eb; border-top: 3px solid #0284c7; border-radius: 10px; padding: 12px 14px;">
      <div style="font-size: 11.5px; color: #6b7280; text-transform: uppercase;">p95 Latency</div>
      <div style="font-size: 22px; font-weight: 800; color: #0369a1; margin: 2px 0;">{p95_factor:.2f}x lower</div>
      <div style="font-size: 12px; color: #374151;">{p95['baseline']:.2f} ms &rarr; {p95['evolved_median']:.2f} ms (-{p95_drop_pct:.1f}%)</div>
    </div>
    <div style="background: #ffffff; border: 1px solid #e5e7eb; border-top: 3px solid #6366f1; border-radius: 10px; padding: 12px 14px;">
      <div style="font-size: 11.5px; color: #6b7280; text-transform: uppercase;">Verification Gate</div>
      <div style="font-size: 22px; font-weight: 800; color: #4338ca; margin: 2px 0;">3 / 3 PASS</div>
      <div style="font-size: 12px; color: #374151;">{blocker_total_ms:,} ms total &middot; {ACTIVE_RUN['repair_turns']} repairs</div>
    </div>
    <div style="background: #ffffff; border: 1px solid #e5e7eb; border-top: 3px solid #0f172a; border-radius: 10px; padding: 12px 14px;">
      <div style="font-size: 11.5px; color: #6b7280; text-transform: uppercase;">Convergence</div>
      <div style="font-size: 22px; font-weight: 800; color: #111827; margin: 2px 0;">Step {ACTIVE_RUN['steps_completed']} / {ACTIVE_RUN['max_steps']}</div>
      <div style="font-size: 12px; color: #374151;">{html.escape(ACTIVE_RUN['status'])} &middot; {ACTIVE_RUN['protected_files_count']} protected</div>
    </div>
  </div>
  <div style="margin-bottom: 12px;">{svg_comparison}</div>
  <div style="margin-bottom: 12px;">{svg_trials}</div>
  <div>{svg_blockers}</div>
</div>
"""

display(HTML(scorecard_html))
print(
    f"Rendered 3 validated SVG scorecard charts for {ACTIVE_RUN['run_id']}: "
    f"ops_per_sec {ops['baseline']:.2f} -> {ops['evolved_median']:.2f} ({ops_speedup:.2f}x), "
    f"p95_ms {p95['baseline']:.2f} -> {p95['evolved_median']:.2f} ({p95_factor:.2f}x), "
    f"blockers {blocker_total_ms}ms."
)


Rendered 3 validated SVG scorecard charts for run_20261123_042237_1dd900: ops_per_sec 141.18 -> 6443.00 (45.64x), p95_ms 7.59 -> 0.19 (39.42x), blockers 1178ms.


## 7. Clean Net Diff & Algorithmic Breakdown (`rsi diff`)

`rsi-cli` records every baseline and promoted commit under hidden Git refs (`refs/rsi/<run_id>/step-0` and `refs/rsi/<run_id>/best`), letting you inspect the clean net diff with `rsi diff <run_id> --view semantic` or `--view unified`.

### Why Baseline (`3724cb21`) Was Slow vs. How Step 1 (`8c25758d`) Achieved 45.64×

| Dimension | Baseline (`3724cb21`) | Promoted Step 1 (`8c25758d`) |
|---|---|---|
| **OpenRouter × Announcement Matching** | `O(D × R)` full `.filter()` scan per announcement, re-running Unicode regex `normalizeStoryTitle` + `significantModelTokens` on every pair | `computeOpenRouterRequiredTokens` collapses all 3 matching rules into one minimal required set; `buildOpenRouterIndex` indexes models by `providerId` + rarest anchor token with 32-bit ASCII bitmasks (`alphaMask`, `digitMask`, `lenMask`) |
| **Tokenization & Allocations** | Rebuilt a fresh 28-word `new Set([...])` + `.split(/\s+/).filter(...)` + template-string `.includes(\` ${token} \`)` on every comparison | Hoisted `MODEL_TOKEN_STOP_WORDS` & `MS_PER_DAY`; char-code ASCII streaming tokenizer (`extractNormalizedTokens` / `findNormalizedStart`), falling back to Unicode normalization only for code points `>= 128` |
| **Top-`k` Selection (`R >= limit`)** | Cloned all `R` models, matched & deduped all `D` announcements, and full-sorted both arrays even when `remainingSlots === 0` (Cases 2 & 3) | `selectTopChronologicalClones` picks & clones only the top `k = limit` OpenRouter models in `O(R log k)`, skips non-official announcements, streams via `enrichProviderFromAnnouncement`, and exits early as soon as `pendingCount === 0` |
| **Announcement Dedupe (`remainingSlots > 0`)** | `O(D²)` scan over prior `extraAnnouncements`, re-tokenizing both sides into fresh `Set`s | Per-provider `ProviderDuplicateIndex` (`tokenPostings` + generation-stamped `seenGen`/`counts` arrays) + `selectTopAnnouncementClones` top-k heap with precomputed `timeMs`/`day`/`official` keys |

**Safety & Scope:** `1 file changed, +1053 / -109` strictly in `packages/shared/src/model-releases.ts`; all **61** protected files untouched; **100% byte-for-byte identical outputs** across all 4 Golden-Master cases plus full unit tests and `tsc --noEmit`.


In [7]:
run_id = ACTIVE_RUN["run_id"]
print("Inspect the net diff locally from any recorded rsi state directory:")
print(f"  rsi diff {run_id} --view semantic")
print(f"  rsi diff {run_id} --view unified")
print(f"\nRecorded Step 1 hypothesis:\n  {ACTIVE_RUN['hypothesis']}")

SANITIZED_DIFF_EXCERPT = """diff --git a/packages/shared/src/model-releases.ts b/packages/shared/src/model-releases.ts
index 3724cb21..8c25758d 100644
--- a/packages/shared/src/model-releases.ts   (Step 0 Baseline: 141.18 ops/s, p95 7.59 ms)
+++ b/packages/shared/src/model-releases.ts   (Step 1 Promoted: 6,443.00 ops/s, p95 0.19 ms)
@@ Baseline hot loop removed: O(D * R) matching + O(D^2) dedupe + full sort @@
-  const mergedOpenRouter = openRouterReleases.map((item) => ({ ...item }));
-  for (const announcement of documentReleases) {
-    const matchedModels = mergedOpenRouter.filter((model) =>
-      announcementMatchesOpenRouterModel(announcement, model),
-    );
-    if (matchedModels.length > 0) {
-      // attach announcementUrl to every matched model
-      continue;
-    }
-    const duplicateAnnouncement = extraAnnouncements.some((existing) => {
-      if (existing.providerId !== announcement.providerId) return false;
-      const existingTokens = new Set(significantModelTokens(existing.modelName));
-      const nextTokens = [...new Set(significantModelTokens(announcement.modelName))];
-      // ... O(D^2) pairwise overlap check
-    });
-  }
-  const selectedOpenRouter = [...mergedOpenRouter]
-    .sort(sortChronologically)
-    .slice(0, limit);
@@ Promoted Step 1: bounded top-k heap + provider/anchor bitmask index + early exit @@
+  if (remainingSlots === 0) {
+    if (openRouterLimit === 0) return [];
+    const selectedOpenRouter = selectTopChronologicalClones(openRouterReleases, openRouterLimit);
+    const { providers, totalUnmatched } = buildOpenRouterIndex(selectedOpenRouter, true);
+    let pendingCount = totalUnmatched;
+    for (let i = 0; i < documentReleases.length && pendingCount > 0; i++) {
+      const announcement = documentReleases[i]!;
+      if (!announcement.announcementUrl) continue;
+      const provIdx = providers.get(announcement.providerId);
+      if (!provIdx || provIdx.remainingUnmatched === 0) continue;
+      pendingCount -= enrichProviderFromAnnouncement(provIdx, announcement);
+    }
+    return selectedOpenRouter;
+  }
+  // General path (remainingSlots > 0): provider + rarest-anchor OpenRouter index,
+  // ProviderDuplicateIndex (tokenPostings + generation-stamped seenGen/counts),
+  // and selectTopAnnouncementClones bounded top-k heap with precomputed sort keys."""

diff_lines_html = []
for line in SANITIZED_DIFF_EXCERPT.splitlines():
    esc = html.escape(line)
    if line.startswith("+++") or line.startswith("---") or line.startswith("diff ") or line.startswith("index "):
        style = "color: #93c5fd; font-weight: 700;"
    elif line.startswith("@@"):
        style = "color: #c4b5fd; background: rgba(139, 92, 246, 0.14);"
    elif line.startswith("+"):
        style = "color: #86efac; background: rgba(34, 197, 94, 0.14);"
    elif line.startswith("-"):
        style = "color: #fca5a5; background: rgba(239, 68, 68, 0.14);"
    else:
        style = "color: #e2e8f0;"
    diff_lines_html.append(f'<div style="padding: 1px 10px; {style}">{esc or "&nbsp;"}</div>')

display(
    HTML(
        f"""
        <div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
                    border: 1px solid #334155; border-radius: 10px; overflow: hidden;
                    margin: 10px 0; max-width: 940px;">
          <div style="background: #1e293b; color: #f8fafc; padding: 10px 14px; font-size: 12.5px;
                      display: flex; justify-content: space-between; align-items: center;">
            <span><strong>Key Sanitized Diff Hunks</strong> &mdash; <code>packages/shared/src/model-releases.ts</code></span>
            <span style="color: #94a3b8;"><code>3724cb21..8c25758d</code> (+1053 / -109)</span>
          </div>
          <div style="background: #0f172a; font-family: ui-monospace, SFMono-Regular, Menlo, monospace;
                      font-size: 12px; line-height: 1.5; padding: 8px 0; overflow-x: auto;">
            {''.join(diff_lines_html)}
          </div>
        </div>
        """
    )
)


Inspect the net diff locally from any recorded rsi state directory:
  rsi diff run_20261123_042237_1dd900 --view semantic
  rsi diff run_20261123_042237_1dd900 --view unified

Recorded Step 1 hypothesis:
  Pre-indexing OpenRouter models by provider and rarest anchor token with a streaming ASCII bitmask filter, bounded top-k heap selection with early exit when OpenRouter releases fill the limit, and per-provider inverted-index announcement deduplication.


## 8. Toggleable GitHub Delivery (`rsi publish`)

Because `examples/ai-newsletter-eval/rsi.yaml` defaults to `publish_on_complete: bundle_only`, promoted optimizations stay safely in local Git refs (`refs/rsi/<run_id>/best`) until you choose how to deliver them:

- **PR-Based (`--mode pull-request`, optional `--auto-merge`):** Squashes promoted steps onto `rsi/<run_id>`, pushes to `origin`, and opens a GitHub Pull Request containing the Baseline vs. Evolved scorecard.
- **Zero-PR (`--mode zero-pr --base-branch main`):** Squashes promoted steps and fast-forward pushes straight to `main`, returning a `/compare/<base>...<head>` audit URL.
- **Bundle-Only (`--mode bundle-only`):** Keeps changes local with zero remote mutations.

Keep `EXECUTE_PUBLISH = False` to preview exact commands safely; flip it to `True` only when you intend to publish a real local run to GitHub.


In [8]:
# Safety guard: leave False for dry preview; flip to True to invoke `rsi publish`.
EXECUTE_PUBLISH = False
PUBLISH_STATE_DIR = os.environ.get("RSI_STATE_DIR", "")

VALID_PUBLISH_MODES = ("pull-request", "zero-pr", "bundle-only", "branch", "tag")


def build_publish_command(
    run_id: str,
    mode: str,
    base_branch: str = "main",
    auto_merge: bool = False,
    remote: str = "origin",
    state_dir: str = "",
    rsi_cmd: str = "rsi",
) -> list[str]:
    if mode not in VALID_PUBLISH_MODES:
        raise ValueError(f"Unsupported publish mode '{mode}'. Choose from: {VALID_PUBLISH_MODES}")
    cmd = [rsi_cmd, "publish", run_id, "--mode", mode]
    if mode == "pull-request" and auto_merge:
        cmd.append("--auto-merge")
    if mode in ("pull-request", "zero-pr") and base_branch:
        cmd.extend(["--base-branch", base_branch])
    if remote and remote != "origin":
        cmd.extend(["--remote", remote])
    if state_dir:
        cmd.extend(["--state-dir", state_dir])
    return cmd


publish_presets = [
    (
        "PR-Based (+ Auto-Merge)",
        "Opens a GitHub PR with the full scorecard and enables --auto --squash merge.",
        build_publish_command(
            ACTIVE_RUN["run_id"],
            mode="pull-request",
            base_branch=str(RUN_PARAMS["base_branch"]),
            auto_merge=True,
        ),
    ),
    (
        "Zero-PR (Direct Trunk Push)",
        "Squashes promoted steps and fast-forward pushes straight onto main with compare URL.",
        build_publish_command(
            ACTIVE_RUN["run_id"],
            mode="zero-pr",
            base_branch=str(RUN_PARAMS["base_branch"]),
        ),
    ),
    (
        "Local Bundle Only",
        "Keeps promoted commit isolated in refs/rsi/<run_id>/best without touching origin.",
        build_publish_command(ACTIVE_RUN["run_id"], mode="bundle-only"),
    ),
]

selected_publish_cmd = build_publish_command(
    ACTIVE_RUN["run_id"],
    mode=str(RUN_PARAMS["publish"]),
    base_branch=str(RUN_PARAMS["base_branch"]),
    auto_merge=bool(RUN_PARAMS["auto_merge"]),
    state_dir=PUBLISH_STATE_DIR,
)

rows_html = "".join(
    f"""
    <tr style="border-top: 1px solid #e5e7eb;">
      <td style="padding: 8px 10px; font-weight: 700; white-space: nowrap;">{html.escape(title)}</td>
      <td style="padding: 8px 10px; color: #4b5563;">{html.escape(desc)}</td>
      <td style="padding: 8px 10px;">
        <code style="background: #f1f5f9; padding: 3px 6px; border-radius: 4px; font-size: 11.5px;">{html.escape(shlex.join(cmd))}</code>
      </td>
    </tr>
    """
    for title, desc, cmd in publish_presets
)

display(
    HTML(
        f"""
        <div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
                    background: #ffffff; color: #1f2937; border: 1px solid #e5e7eb;
                    border-radius: 10px; padding: 14px 16px; margin: 8px 0; max-width: 940px;">
          <div style="font-size: 14px; font-weight: 700; margin-bottom: 8px;">
            GitHub Delivery Modes (<code>rsi publish</code>)
          </div>
          <table style="width: 100%; border-collapse: collapse; font-size: 12.5px; margin-bottom: 10px;">
            <thead>
              <tr style="background: #f9fafb; text-align: left;">
                <th style="padding: 8px 10px;">Mode</th>
                <th style="padding: 8px 10px;">Behavior</th>
                <th style="padding: 8px 10px;">CLI Command</th>
              </tr>
            </thead>
            <tbody>{rows_html}</tbody>
          </table>
          <div style="font-size: 12px; color: #4b5563; margin-bottom: 4px;">
            Selected command from Section 3 <code>RUN_PARAMS['publish'] = '{html.escape(str(RUN_PARAMS['publish']))}'</code>:
          </div>
          <pre style="margin: 0; padding: 10px 12px; background: #0f172a; color: #e2e8f0;
                      border-radius: 6px; font-size: 12px; overflow-x: auto;">{html.escape(shlex.join(selected_publish_cmd))}</pre>
        </div>
        """
    )
)

if EXECUTE_PUBLISH:
    if not RSI_BIN:
        raise RuntimeError("rsi binary not found; build with `cargo build --release` first.")
    exec_cmd = build_publish_command(
        ACTIVE_RUN["run_id"],
        mode=str(RUN_PARAMS["publish"]),
        base_branch=str(RUN_PARAMS["base_branch"]),
        auto_merge=bool(RUN_PARAMS["auto_merge"]),
        state_dir=PUBLISH_STATE_DIR,
        rsi_cmd=RSI_BIN,
    )
    proc = subprocess.run(
        exec_cmd,
        cwd=REPO_ROOT,
        stdin=subprocess.DEVNULL,
        capture_output=True,
        text=True,
        check=False,
    )
    print(f"--- Live `rsi publish` output (exit {proc.returncode}) ---")
    print(sanitize_text(proc.stdout.strip()))
    if proc.stderr.strip():
        print("\n[stderr]\n" + sanitize_text(proc.stderr.strip()))
else:
    print("EXECUTE_PUBLISH = False -> preview only (no remote Git pushes or GitHub PRs executed).")


Mode,Behavior,CLI Command
PR-Based (+ Auto-Merge),Opens a GitHub PR with the full scorecard and enables --auto --squash merge.,rsi publish run_20261123_042237_1dd900 --mode pull-request --auto-merge --base-branch main
Zero-PR (Direct Trunk Push),Squashes promoted steps and fast-forward pushes straight onto main with compare URL.,rsi publish run_20261123_042237_1dd900 --mode zero-pr --base-branch main
Local Bundle Only,Keeps promoted commit isolated in refs/rsi/<run_id>/best without touching origin.,rsi publish run_20261123_042237_1dd900 --mode bundle-only


EXECUTE_PUBLISH = False -> preview only (no remote Git pushes or GitHub PRs executed).
